# Chapter 12
## Two-Dimensional Bifurcation Analysis
- Code by : [Abolfazl Ziaeemehr](https://github.com/Ziaeemehr)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ITNG/ModelingNeuralDynamics/blob/main/python/chapter12.ipynb)

## About this chapter

The reduced RTM model ($m=m_\infty(v)$, $h=1-n$) is analyzed as a
two-dimensional dynamical system in $(v,n)$: its fixed points are tracked
as a function of $I_{ext}$ and classified by the eigenvalues of the
Jacobian (node, saddle, or spiral), and an invariant cycle bounded by two
fixed points is traced out at $I_{ext}=0$.

$$
\dot v = \big(g_{Na}m_\infty(v)^3(1-n)(E_{Na}-v) + g_K n^4(E_K-v)
+ g_L(E_L-v) + I_{ext}\big)/C,\qquad
\dot n = \alpha_n(v)(1-n)-\beta_n(v)n.
$$

See [`README.md`](chapter12.md) for the full guide, including suggested
order and related chapters.

In [ ]:
import subprocess
import sys
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "modelingneuraldynamics"], check=True)

In [ ]:
import numpy as np
from numpy import exp
import matplotlib.pyplot as plt
from ipywidgets import interact
from mnd.core import draw_arrow, alpha_h, alpha_m, alpha_n, beta_h, beta_m, beta_n, h_inf, m_inf, n_inf

## Fixed-Point Bifurcation Diagram

Classifies every fixed point of the reduced RTM model, for a range of
$I_{ext}$, by the eigenvalues of its Jacobian.

In [ ]:
def simulate_rtm_2d_fp(c=1.0, g_k=80.0, g_na=100.0, g_l=0.1,
                        v_k=-100.0, v_na=50.0, v_l=-67.0,
                        i_ext_vec=None):
    if i_ext_vec is None:
        i_ext_vec = np.arange(1001) / 1000 * 0.2

    def alpha_m_p(v):
        num, den = 0.32 * (v + 54), 1 - exp(-(v + 54) / 4)
        num_p, den_p = 0.32, exp(-(v + 54) / 4) / 4
        return (den * num_p - num * den_p) / den ** 2

    def beta_m_p(v):
        num, den = 0.28 * (v + 27), exp((v + 27) / 5) - 1
        num_p, den_p = 0.28, exp((v + 27) / 5) / 5
        return (den * num_p - num * den_p) / den ** 2

    def alpha_n_local(v):
        # matlab/12/RTM_2D_FP/alpha_n.m has a removable singularity at v=-52
        # and no guard for it either -- NaN there, same as here, harmless
        # since NaN comparisons are just False in both languages
        with np.errstate(divide='ignore', invalid='ignore'):
            return 0.032 * (v + 52) / (1 - exp(-(v + 52) / 5))

    def alpha_n_p(v):
        num, den = 0.032 * (v + 52), 1 - exp(-(v + 52) / 5)
        num_p, den_p = 0.032, exp(-(v + 52) / 5) / 5
        return (den * num_p - num * den_p) / den ** 2

    def beta_n_p(v):
        return -beta_n(v) / 40

    def m_inf_p(v):
        num = (alpha_m(v) + beta_m(v)) * alpha_m_p(v)
        num = num - alpha_m(v) * (alpha_m_p(v) + beta_m_p(v))
        return num / (alpha_m(v) + beta_m(v)) ** 2

    def n_inf_local(v):
        return alpha_n_local(v) / (alpha_n_local(v) + beta_n(v))

    def f(v, i_ext):
        """dv/dt of the reduced (m=m_inf(v), h=1-n_inf(v)) RTM model"""
        return (g_na * m_inf(v) ** 3 * (1 - n_inf_local(v)) * (v_na - v)
                + g_k * n_inf_local(v) ** 4 * (v_k - v) + g_l * (v_l - v) + i_ext)

    def find_fixed_points(i_ext):
        v_min = min(v_k, v_l + i_ext / g_l)
        v_max = max(v_na, v_l + i_ext / g_l)
        n_v = 1000
        v_vec = v_min + np.arange(n_v + 1) / n_v * (v_max - v_min)
        f_vec = f(v_vec, i_ext)

        roots = []
        for i in np.where(f_vec[:-1] * f_vec[1:] <= 0)[0]:
            a, b_ = v_vec[i], v_vec[i + 1]
            while b_ - a > 1e-10:
                mid = (a + b_) / 2
                if f(mid, i_ext) * f(a, i_ext) <= 0:
                    b_ = mid
                else:
                    a = mid
            roots.append((a + b_) / 2)
        return roots

    def jacobian(v, n):
        j00 = (g_na * 3 * m_inf(v) ** 2 * m_inf_p(v) * (1 - n) * (v_na - v)
               - g_na * m_inf(v) ** 3 * (1 - n) - g_k * n ** 4 - g_l)
        j01 = -g_na * m_inf(v) ** 3 * (v_na - v) + g_k * 4 * n ** 3 * (v_k - v)
        j10 = alpha_n_p(v) * (1 - n) - beta_n_p(v) * n
        j11 = -alpha_n_local(v) - beta_n(v)
        return np.array([[j00, j01], [j10, j11]]) / c

    points = {'g': [], 'r': [], 'k': [], 'm': [], 'b': []}
    for i_ext in i_ext_vec:
        for v in find_fixed_points(i_ext):
            n = n_inf_local(v)
            e = np.linalg.eigvals(jacobian(v, n))
            if abs(e[0].imag) > 1e-4:
                if e[0].real > 0:
                    points['g'].append((i_ext, v))
                if e[0].real < 0:
                    points['r'].append((i_ext, v))
            if e[0].real < 0 and e[1].real < 0:
                points['k'].append((i_ext, v))
            if e[0].real * e[1].real < 0:
                points['m'].append((i_ext, v))
            if e[0].real > 0 and e[1].real > 0:
                points['b'].append((i_ext, v))
    return points, i_ext_vec


def plot_rtm_2d_fp(points, i_ext_vec):
    plt.figure(figsize=(7, 7))
    for color, pts in points.items():
        if pts:
            i_pts, v_pts = zip(*pts)
            plt.plot(i_pts, v_pts, '.' + color, markersize=3)

    plt.xlim(min(i_ext_vec), max(i_ext_vec))
    plt.ylim(-70, -30)
    plt.xlabel(r'$I$ [$\mu$A/cm$^2$]')
    plt.ylabel(r'$v_\ast$ [mV]')
    plt.tight_layout()
    plt.show()

In [ ]:
plot_rtm_2d_fp(*simulate_rtm_2d_fp())

## RTM Invariant Cycle

At $I_{ext}=0$, two fixed points (one stable node, one saddle) bound an
invariant cycle in the $(v,n)$ phase plane; nearby trajectories spiral onto
it from either side.

In [ ]:
def simulate_rtm_trajectory(v0, n0, i_ext, t_final, c=1.0, g_k=80.0, g_na=100.0, g_l=0.1,
                             v_k=-100.0, v_na=50.0, v_l=-67.0, dt=0.005):
    """reduced RTM model: m=m_inf(v), h=1-n"""
    m_steps = round(t_final / dt)
    v, n = np.zeros(m_steps + 1), np.zeros(m_steps + 1)
    v[0], n[0] = v0, n0
    for k in range(m_steps):
        h = 1 - n[k]
        v_inc = (g_k * n[k] ** 4 * (v_k - v[k]) + g_na * m_inf(v[k]) ** 3 * h * (v_na - v[k])
                 + g_l * (v_l - v[k]) + i_ext) / c
        n_inc = alpha_n(v[k]) * (1 - n[k]) - beta_n(v[k]) * n[k]
        v_tmp = v[k] + dt / 2 * v_inc
        n_tmp = n[k] + dt / 2 * n_inc
        h_tmp = 1 - n_tmp
        v_inc = (g_k * n_tmp ** 4 * (v_k - v_tmp) + g_na * m_inf(v_tmp) ** 3 * h_tmp * (v_na - v_tmp)
                 + g_l * (v_l - v_tmp) + i_ext) / c
        n_inc = alpha_n(v_tmp) * (1 - n_tmp) - beta_n(v_tmp) * n_tmp
        v[k + 1] = v[k] + dt * v_inc
        n[k + 1] = n[k] + dt * n_inc
    return v, n


def find_root(f, a, b, tol):
    while b - a > tol:
        c_ = (a + b) / 2
        if f(c_) * f(a) > 0:
            a = c_
        else:
            b = c_
    return c_


def rtm_2d_invariant_cycle_roots(g_k=80.0, g_na=100.0, g_l=0.1, v_k=-100.0, v_na=50.0, v_l=-67.0, i_ext=0.0):
    def f(v):
        # note: h_inf(v) here, not the 1-n reduction simulate_rtm_trajectory
        # uses -- matches matlab/12/RTM_2D_INVARIANT_CYCLE/make_figure.m exactly
        return (g_k * n_inf(v) ** 4 * (v_k - v) + g_na * m_inf(v) ** 3 * h_inf(v) * (v_na - v)
                + g_l * (v_l - v) + i_ext)

    v_star = find_root(f, -63, -62, 1e-14)
    v_0 = find_root(f, -75, -65, 1e-12)
    return v_star, v_0


def mark_crossings(ax, v, n, condition, ax_xlim, ax_ylim, color='k'):
    for k in np.where(condition)[0]:
        vec = [v[k + 1] - v[k], n[k + 1] - n[k]]
        draw_arrow(ax, ax_xlim, ax_ylim, v[k], n[k], vec, epsilon=0.1, width=2, color=color)


def plot_rtm_2d_invariant_cycle():
    frame_xlim, frame_ylim = [-105, 55], [-0.05, 0.8]
    xlim, ylim = [-75, -50], [0, 0.15]

    fig, axes = plt.subplots(1, 3, figsize=(15, 5.5))

    v, n = simulate_rtm_trajectory(-75.582445796204553, 0.005935897840905, i_ext=1.0, t_final=30.0)
    mark_crossings(axes[0], v, n, (v[:-1] < 0) & (v[1:] > 0), frame_xlim, frame_ylim)
    mark_crossings(axes[0], v, n, (v[:-1] > -30) & (v[1:] < -30), frame_xlim, frame_ylim)
    axes[0].plot(v, n, color='k', linewidth=2)
    axes[0].set_xlim(*frame_xlim)
    axes[0].set_ylim(*frame_ylim)
    axes[0].set_box_aspect(1)
    axes[0].set_xlabel('$v$ [mV]')
    axes[0].set_ylabel('$n$')

    v_star, v_0 = rtm_2d_invariant_cycle_roots()
    n_star = n_inf(v_star)
    n_0 = n_inf(v_0)

    v, n = simulate_rtm_trajectory(v_star + 0.5, n_star + 0.005, i_ext=0.0, t_final=300.0)
    mark_crossings(axes[1], v, n, (v[:-1] < 0) & (v[1:] > 0), frame_xlim, frame_ylim)
    mark_crossings(axes[1], v, n, (v[:-1] > -30) & (v[1:] < -30), frame_xlim, frame_ylim)
    axes[1].plot(v, n, color='k', linewidth=2)
    axes[1].set_xlim(*frame_xlim)
    axes[1].set_ylim(*frame_ylim)
    axes[1].set_box_aspect(1)
    axes[1].set_xlabel('$v$ [mV]')

    axes[2].plot(v, n, color='k', linewidth=2)
    ind = np.where((v[:-1] < -55) & (v[1:] >= -55))[0][0]
    vec = [v[ind + 1] - v[ind], n[ind + 1] - n[ind]]
    draw_arrow(axes[2], xlim, ylim, v[ind], n[ind], vec, epsilon=0.1, width=2)
    ind = np.where((v[:-1] < -70) & (v[1:] >= -70))[0][0]
    vec = [v[ind + 1] - v[ind], n[ind + 1] - n[ind]]
    draw_arrow(axes[2], xlim, ylim, v[ind], n[ind], vec, epsilon=0.1, width=2)

    v2, n2 = simulate_rtm_trajectory(v_star - 0.5, n_star - 0.005, i_ext=0.0, t_final=300.0)
    mark_crossings(axes[2], v2, n2, (v2[:-1] > -65) & (v2[1:] < -65), xlim, ylim, color='r')
    axes[2].plot(v2, n2, color='r', linewidth=2)
    axes[2].plot(v_star, n_star, 'ok', markersize=8, markerfacecolor='w')
    axes[2].plot(v_0, n_0, 'ok', markersize=8, markerfacecolor='k')
    axes[2].set_xlim(*xlim)
    axes[2].set_ylim(*ylim)
    axes[2].set_box_aspect(1)
    axes[2].set_xlabel('$v$ [mV]')

    plt.tight_layout()
    plt.show()

In [ ]:
plot_rtm_2d_invariant_cycle()